# 03 — Offsets, tenors et dates de règlement

Trois outils qui s'empilent : l'objet `BDay` pour écrire `date + BDay(3)`, la grammaire de
tenors pour `"1Y+2B"`, et les lags de règlement pour « ça règle quand ? ».

In [1]:
from datetime import date, datetime

import numpy as np
import pandas as pd

import better_calendar as bcal
from better_calendar import BDay, Calendar, Roll

## 1. `BDay` : l'offset comme objet

In [2]:
print(date(2026, 7, 31) + BDay(1))                   # vendredi -> lundi
print(date(2026, 8, 3) - BDay(1))
print("2026-07-02" + BDay(1, cal="XNYS"))            # saute le 3 juillet
print(date(2026, 7, 31) + BDay(1) * 5)               # multipliable
print(-BDay(2))

2026-08-03
2026-07-31
2026-07-06
2026-08-07
BDay(n=-2, cal=None, roll=<Roll.FOLLOWING: 'following'>)


Il marche sur **tous** les types d'entrée, et sur les conteneurs pandas :

In [3]:
serie = pd.Series(pd.DatetimeIndex(["2026-07-31", "2026-08-03", "2026-12-24"]))
resultat = serie + BDay(3, cal="XNYS")
pd.DataFrame({"départ": serie, "+3 jours ouvrés NYSE": resultat})

,départ,+3 jours ouvrés NYSE
0,2026-07-31,2026-08-05
1,2026-08-03,2026-08-06
2,2026-12-24,2026-12-30


Les conteneurs sont la partie délicate. Quand on écrit `serie + BDay(3)`, pandas ne nous
passe **pas** la `Series` : il déballe le tableau sous-jacent, appelle `__radd__` avec, puis
reconstruit le conteneur à partir de ce qu'on renvoie. Il faut donc renvoyer quelque chose
que pandas sait ré-emballer, **et** qui porte le fuseau si l'entrée en avait un.

Pour un tableau numpy nu, il faut en plus désactiver la machinerie ufunc, sinon
l'opération meurt à l'intérieur de numpy.

In [4]:
tableau = np.array(["2026-07-31", "2026-08-03"], dtype="datetime64[D]")
print("numpy   :", (tableau + BDay(1)).strftime("%Y-%m-%d").tolist())

index = pd.DatetimeIndex(["2026-07-31"]).tz_localize("Europe/Paris")
decale = index + BDay(1, cal=Calendar("paris", tz="Europe/Paris"))
print("tz-aware:", decale.strftime("%Y-%m-%d %H:%M %Z").tolist(), "|", decale.dtype)

numpy   : ['2026-08-03', '2026-08-04']
tz-aware: ['2026-08-03 00:00 CEST'] | datetime64[ns, Europe/Paris]


En pratique, `cal.offset(serie, 3)` reste la forme recommandée : même réponse, chemin plus
court, pas de dispatch d'opérateur à démêler. `BDay` existe pour les endroits où un
**objet** offset se lit mieux — un argument par défaut, une valeur de configuration.

In [5]:
nyse = bcal.get("XNYS")
pd.testing.assert_series_equal(serie + BDay(3, cal="XNYS"), pd.Series(nyse.offset(serie, 3)))
print("les deux formes donnent le même résultat")

les deux formes donnent le même résultat


### Interop pandas : `to_pandas_offset` et son écart

Pour les machineries pandas qui exigent un vrai `DateOffset` (`date_range`, `resample`) :

In [6]:
offset_pandas = nyse.to_pandas_offset()
pd.date_range("2026-07-01", "2026-07-10", freq=offset_pandas)

DatetimeIndex(['2026-07-01', '2026-07-02', '2026-07-06', '2026-07-07',
               '2026-07-08', '2026-07-09', '2026-07-10'],
              dtype='datetime64[ns]', freq='C')

**Attention, écart réel** : `to_pandas_offset()` et `cal.offset()` ne donnent pas la même
réponse quand le départ **n'est pas** un jour ouvré. Nous normalisons puis avançons
(sémantique `numpy.busday_offset`) ; pandas compte la normalisation comme le déplacement.

In [7]:
lignes = []
for depart in ("2026-07-31", "2026-07-02", "2026-08-01", "2026-07-03"):
    lignes.append(
        {
            "départ": depart,
            "ouvré": nyse.is_bday(depart),
            "cal.offset(d, 1)": nyse.offset(depart, 1),
            "d + CustomBusinessDay": str((pd.Timestamp(depart) + offset_pandas).date()),
        }
    )
table = pd.DataFrame(lignes)
table["accord"] = table["cal.offset(d, 1)"] == table["d + CustomBusinessDay"]
table

,départ,ouvré,"cal.offset(d, 1)",d + CustomBusinessDay,accord
0,2026-07-31,True,2026-08-03,2026-08-03,True
1,2026-07-02,True,2026-07-06,2026-07-06,True
2,2026-08-01,False,2026-08-04,2026-08-03,False
3,2026-07-03,False,2026-07-07,2026-07-06,False


Aucun des deux n'a tort. Mais mélanger les deux dans un même pipeline finit mal, donc
mieux vaut partir d'un jour ouvré ou rester sur `cal.offset` et expliciter `roll`.

### L'accesseur `.cal`

Importer le module d'intégration enregistre un accesseur sur `Series` et `Index` :

In [8]:
import better_calendar.integrations.pandas_  # enregistre .cal

trades = pd.DataFrame({"traded": pd.to_datetime(["2026-07-02", "2026-08-01", "2026-12-24"])})
trades["ouvré"] = trades["traded"].cal.is_bday("XNYS")
trades["règlement"] = trades["traded"].cal.offset(2, "XNYS")
trades["fin de mois"] = trades["traded"].cal.add_tenor("1M", "XNYS", roll="MF")
trades

,traded,ouvré,règlement,fin de mois
0,2026-07-02,True,2026-07-07,2026-08-03
1,2026-08-01,False,2026-08-05,2026-09-01
2,2026-12-24,True,2026-12-29,2027-01-25


## 2. Tenors

Grammaire : `terme (('+' | '-') terme)*` où un terme est `[-] ENTIER unité`, avec les
unités `D` (jours calendaires), `B` (jours ouvrés), `W` (semaines), `M` (mois), `Y` (années).

In [9]:
depart = "2026-07-31"     # un vendredi
pd.DataFrame(
    [{"tenor": t, "résultat": bcal.add_tenor(depart, t, cal="XNYS")}
     for t in ("1D", "3D", "1W", "2W", "1B", "5B", "1M", "3M", "1Y", "1Y+2B", "1M-1B")]
).set_index("tenor")

,résultat
tenor,
1D,2026-08-01
3D,2026-08-03
1W,2026-08-07
2W,2026-08-14
1B,2026-08-03
5B,2026-08-07
1M,2026-08-31
3M,2026-10-31
1Y,2027-07-31


`parse_tenor` expose la structure analysée, utile pour valider une configuration avant de
l'exécuter — et le résultat est mémoïsé, parce que les tenors arrivent de fichiers de
config dans des boucles chaudes.

In [10]:
analyse = bcal.parse_tenor("1Y-2B")
print("termes          :", analyse.terms)
print("besoin calendrier:", analyse.needs_calendar)
print("mémoïsé         :", bcal.parse_tenor("1Y-2B") is analyse)
print("sans terme B    :", bcal.parse_tenor("3M").needs_calendar)

termes          : (TenorTerm(count=1, unit='Y'), TenorTerm(count=-2, unit='B'))
besoin calendrier: True
mémoïsé         : True
sans terme B    : False


### Les deux règles de fin de mois, qu'il ne faut surtout pas confondre

C'est ici que vivent les bugs d'un jour.

- **Le clamping** est inconditionnel : le 31 janvier + 1 mois donne le 28 février, parce
  que le 31 février n'existe pas. Rien d'optionnel là-dedans.
- **La règle EOM** est optionnelle (`eom=True`) : si la date de départ est le **dernier
  jour** de son mois, le résultat est le dernier jour du mois cible.

In [11]:
cas = ["2026-01-31", "2026-02-28", "2026-04-30", "2026-02-27", "2026-07-15", "2024-02-29"]
pd.DataFrame(
    [
        {
            "départ": d,
            "fin de mois ?": pd.Timestamp(d).is_month_end,
            "+1M": bcal.add_tenor(d, "1M"),
            "+1M (eom=True)": bcal.add_tenor(d, "1M", eom=True),
        }
        for d in cas
    ]
).set_index("départ")

,fin de mois ?,+1M,+1M (eom=True)
départ,,,
2026-01-31,True,2026-02-28,2026-02-28
2026-02-28,True,2026-03-28,2026-03-31
2026-04-30,True,2026-05-30,2026-05-31
2026-02-27,False,2026-03-27,2026-03-27
2026-07-15,False,2026-08-15,2026-08-15
2024-02-29,True,2024-03-29,2024-03-31


Les deux dernières lignes montrent que la règle EOM ne se déclenche **que** si le départ
est en fin de mois : le 27 février et le 15 juillet ne bougent pas.

Le clamping, lui, n'est pas réversible — et c'est voulu :

In [12]:
aller = bcal.add_tenor("2026-01-31", "1M")
retour = bcal.add_tenor(aller, "-1M")
print(f"2026-01-31 +1M -> {aller} ; puis -1M -> {retour}")
print("de l'information a été perdue au clamping, ce qui est le comportement correct")

2026-01-31 +1M -> 2026-02-28 ; puis -1M -> 2026-01-28
de l'information a été perdue au clamping, ce qui est le comportement correct


### Les termes s'appliquent de gauche à droite

`"1M+2B"` n'est **pas** `"2B+1M"` en général : ajouter un mois à un vendredi et ajouter un
mois au mardi suivant ne tombent pas dans la même semaine.

In [13]:
depart = "2026-01-30"   # un vendredi
print(f'{depart}  "1M+2B" -> {bcal.add_tenor(depart, "1M+2B")}')
print(f'{depart}  "2B+1M" -> {bcal.add_tenor(depart, "2B+1M")}')

2026-01-30  "1M+2B" -> 2026-03-04
2026-01-30  "2B+1M" -> 2026-03-03


Une erreur de syntaxe désigne le fragment fautif :

In [14]:
for mauvais in ("3Q", "3M5D", "1.5M"):
    try:
        bcal.add_tenor("2026-01-01", mauvais)
    except bcal.TenorParseError as exc:
        print(f"{mauvais!r:10s} -> {exc}")

'3Q'       -> Cannot parse tenor '3Q': unknown unit at '3Q'. Expected terms like '3M', '2B', '-1Y+2B' with units D, B, W, M or Y.
'3M5D'     -> Cannot parse tenor '3M5D': expected '+' or '-' between terms at '5D'. Expected terms like '3M', '2B', '-1Y+2B' with units D, B, W, M or Y.
'1.5M'     -> Cannot parse tenor '1.5M': unknown unit at '1.5M'. Expected terms like '3M', '2B', '-1Y+2B' with units D, B, W, M or Y.


Le roll par défaut d'un tenor est `NONE` : un tenor est une **période**, l'ajuster est une
décision distincte qu'on prend explicitement.

In [15]:
print("brut       :", bcal.add_tenor("2026-04-30", "1M", eom=True))                    # dimanche
print("ajusté MF  :", bcal.add_tenor("2026-04-30", "1M", eom=True, roll=Roll.MODIFIED_FOLLOWING))

brut       : 2026-05-31
ajusté MF  : 2026-05-29


## 3. Dates de règlement

`spot(d, devise)` répond à « si je traite aujourd'hui, ça règle quand ». Le calendrier
par défaut vient de la devise via la table d'alias.

In [16]:
pd.DataFrame(
    [
        {
            "devise": c,
            "lag (j. ouvrés)": bcal.spot_lag(c),
            "calendrier": bcal.get(c).name,
            "spot du 2026-07-31": bcal.spot("2026-07-31", c),
        }
        for c in ("EUR", "USD", "GBP", "CAD", "JPY", "CHF", "TRY")
    ]
).set_index("devise")

,lag (j. ouvrés),calendrier,spot du 2026-07-31
devise,,,
EUR,2,fin:TARGET2,2026-08-04
USD,2,fin:NYB,2026-08-04
GBP,0,fin:LNB,2026-07-31
CAD,1,fin:TRB,2026-08-04
JPY,2,fin:TKB,2026-08-04
CHF,2,fin:ZUB,2026-08-04
TRY,0,ql:Turkey,2026-07-31


Le sterling règle le jour même (T+0) : ce sont des conventions **monétaires / dépôt**, pas
des conventions de spot FX — lesquelles sont une propriété de la *paire*, pas d'une devise.

Le dollar canadien est à T+1 mais tombe sur le 4 août : le 3 août est le Civic Holiday à
Toronto. Le calendrier fait son travail.

In [17]:
print("CAD T+1 depuis vendredi 31 juillet :", bcal.spot("2026-07-31", "CAD"))
print("le 3 août est-il ouvré à Toronto ? ", bcal.get("CAD").is_bday("2026-08-03"))

CAD T+1 depuis vendredi 31 juillet : 2026-08-04
le 3 août est-il ouvré à Toronto ?  False


Pour un trade cross-devises qui doit régler dans **deux** places à la fois, on passe le
composite :

In [18]:
deux_places = bcal.get("EUR") & bcal.get("USD")
print("calendrier :", deux_places.name)
print("EUR seul   :", bcal.spot("2026-07-01", "EUR"))
print("EUR & USD  :", bcal.spot("2026-07-01", "EUR", cal=deux_places), " (saute le 3 juillet)")

calendrier : (fin:TARGET2 & fin:NYB)
EUR seul   : 2026-07-03
EUR & USD  : 2026-07-06  (saute le 3 juillet)


La table des lags est un fichier de données, pas du code — un desk dont la convention
diffère corrige une ligne sans release. Elle est en lecture seule à l'exécution, parce
qu'une mutation déplacerait silencieusement toutes les dates de règlement suivantes.

In [19]:
print("devises connues :", ", ".join(sorted(bcal.SPOT_LAG)))
try:
    bcal.SPOT_LAG["EUR"] = 99
except TypeError as exc:
    print("\nmutation refusée :", exc)

devises connues : AUD, CAD, CHF, CZK, DKK, EUR, GBP, HKD, HUF, JPY, MXN, NOK, NZD, PLN, SEK, SGD, TRY, USD, ZAR

mutation refusée : 'mappingproxy' object does not support item assignment


## Récapitulatif

| Appel | Rôle |
|---|---|
| `BDay(n, cal=, roll=)` | offset objet, `d + BDay(3)` |
| `cal.offset(serie, n)` | la forme recommandée pour un conteneur |
| `cal.to_pandas_offset()` | vrai `DateOffset` pour `date_range` / `resample` |
| `.cal` accessor | `serie.cal.offset(2, "XNYS")` |
| `add_tenor(d, "1Y+2B", eom=)` | grammaire de tenors, gauche à droite |
| `spot(d, ccy, cal=)` | date de règlement |
| `SPOT_LAG`, `spot_lag(ccy)` | la table des lags |

**Suite :** [04 — Récurrences et échéanciers](04-recurrences-et-echeanciers.ipynb)